Documentação Metodológica: Cálculo do Índice de Vulnerabilidade Social (IVS)

Este notebook realiza a transformação das variáveis brutas do Censo 2022 em indicadores proporcionais de vulnerabilidade, numa escala de 0 a 1 (onde 1 representa a vulnerabilidade máxima).
1. Filtros de Exclusão

Setores censitários com População igual a zero ou sem Domicílios Particulares Permanentes Ocupados são removidos da análise para evitar distorções matemáticas (divisão por zero).
2. Indicadores Proporcionais (Cálculo Direto)

A lógica base para Saneamento, Educação e Cor/Raça é: Risco / Universo Total.

    Água Inadequada: (Poço + Nascente + Carro-pipa + Chuva + Rio + Outros) / Total de Domicílios.

    Esgoto Inadequado: (Fossa rudimentar + Vala + Rio + Outros + Inexistente) / Total de Domicílios.

    Lixo Inadequado: (Caçamba + Queimado + Enterrado + Terreno Baldio + Outros) / Total de Domicílios.

    Analfabetismo: Pessoas analfabetas (15 anos ou mais) / População total (15 anos ou mais).

    Cor ou Raça: (Pretos + Pardos + Indígenas) / População total do setor.

3. Indicador de Habitação (Densidade Domiciliar)

Utiliza-se diretamente a variável v0005 do IBGE, que já entrega o cálculo exato da média de moradores por domicílio, expurgando automaticamente os moradores de domicílios coletivos (presídios, asilos, etc.).
4. Indicador de Renda (Normalização Min-Max Invertida)

Como não há dados de contagem por faixas de salário mínimo, utiliza-se a Renda Média do setor (V06004). Para transformar este valor em dinheiro num índice de risco (0 a 1), aplica-se a fórmula:

    Índice de Renda = (Renda Máxima do Município - Renda do Setor) / (Renda Máxima - Renda Mínima)
    Desta forma, os setores mais pobres recebem valor próximo de 1, e os mais ricos valor próximo de 0. Setores sem informação de renda recebem o risco máximo (1.0).

In [ ]:
# Importa bibliotecas para manipulação de dados e banco de dados
import pandas as pd        # pandas para DataFrame
import sqlite3             # sqlite3 para conexão com banco SQLite
import numpy as np         # numpy para operações numéricas

# Mensagem inicial
print("Iniciando a extração dos dados do banco essencial...")

# Define caminhos para o banco de dados
caminho_bd = '../banco_de_dados/'
caminho_sqlite = caminho_bd + 'Banco_IVS_Essencial.db'

# Abre conexão com o banco SQLite
conexao = sqlite3.connect(caminho_sqlite)

# Extrai a tabela base_ivs_bruta para um DataFrame
df_ivs = pd.read_sql_query("SELECT * FROM base_ivs_bruta", conexao)

# Fecha a conexão com o banco
conexao.close()

# Exibe quantidade de linhas e colunas extraídas
print("Dados extraídos com sucesso! Linhas:", df_ivs.shape[0], "Colunas:", df_ivs.shape[1])

Iniciando a extração dos dados do banco essencial...
Dados extraídos com sucesso! Linhas: 468099 Colunas: 31


In [ ]:
# Aplica limpeza de texto e filtros de exclusão populacional
print("Aplicando limpeza de texto e filtros de exclusão populacional...")

# Lista de colunas não numéricas
colunas_nao_numericas = ['CD_SETOR', 'NM_MUN', 'NM_BAIRRO', 'SITUACAO']
# Seleciona colunas numéricas
colunas_numericas = [col for col in df_ivs.columns if col not in colunas_nao_numericas]

# Converte tudo para número, forçando erros a virarem NaN, e depois preenche com 0
for col in colunas_numericas:
    df_ivs[col] = pd.to_numeric(df_ivs[col], errors='coerce')

# Preenche NaN com 0
df_ivs = df_ivs.fillna(0)

# Aplica a regra de exclusão metodológica
filtro_populacao = df_ivs['v0001'] > 0    # Filtra setores com população > 0
filtro_domicilios = df_ivs['V00001'] > 0  # Filtra setores com domicílios > 0
df_limpo = df_ivs[filtro_populacao & filtro_domicilios].copy()  # Aplica ambos os filtros

# Exibe estatísticas dos setores
print("Setores originais:", len(df_ivs))
print("Setores válidos para o cálculo:", len(df_limpo))
print("Setores excluídos (vazios/sigilosos):", len(df_ivs) - len(df_limpo))

Aplicando limpeza de texto e filtros de exclusão populacional...
Setores originais: 468099
Setores válidos para o cálculo: 450088
Setores excluídos (vazios/sigilosos): 18011


In [ ]:
# Calcula indicadores proporcionais de vulnerabilidade para saneamento, educação, raça e habitação
print("Calculando as dimensões de Saneamento, Educação, Raça e Habitação...")

# 1. Saneamento: Água Inadequada
soma_agua_ruim = df_limpo[['V00112', 'V00113', 'V00114', 'V00115', 'V00116', 'V00117', 'V00118']].sum(axis=1)  # Soma fontes inadequadas de água
df_limpo['ind_agua_inadequada'] = soma_agua_ruim / df_limpo['V00001']  # Calcula proporção de água inadequada

# 2. Saneamento: Esgoto Inadequado
soma_esgoto_ruim = df_limpo[['V00312', 'V00313', 'V00314', 'V00315', 'V00316']].sum(axis=1)  # Soma esgoto inadequado
df_limpo['ind_esgoto_inadequado'] = soma_esgoto_ruim / df_limpo['V00001']  # Proporção de esgoto inadequado

# 3. Saneamento: Lixo Inadequado
soma_lixo_ruim = df_limpo[['V00398', 'V00399', 'V00400', 'V00401', 'V00402']].sum(axis=1)  # Soma lixo inadequado
df_limpo['ind_lixo_inadequado'] = soma_lixo_ruim / df_limpo['V00001']  # Proporção de lixo inadequado

# 4. Educação: Analfabetismo
df_limpo['ind_analfabetismo'] = np.where(df_limpo['V00900'] > 0, df_limpo['V00901'] / df_limpo['V00900'], 0)  # Proporção de analfabetismo

# 5. Social: Cor ou Raça
soma_vulneraveis = df_limpo[['V01318', 'V01320', 'V01321']].sum(axis=1)  # Soma pretos, pardos e indígenas
df_limpo['ind_cor_raca'] = soma_vulneraveis / df_limpo['v0001']          # Proporção de vulneráveis por cor/raça

# 6. Habitação: Densidade Habitacional (variável pronta do IBGE)
df_limpo['ind_densidade_habitacional'] = df_limpo['v0005']               # Densidade habitacional

# Garante que nenhum índice passe de 1.0 devido a arredondamentos da base do IBGE
colunas_ind = ['ind_agua_inadequada', 'ind_esgoto_inadequado', 'ind_lixo_inadequado', 'ind_analfabetismo', 'ind_cor_raca']  # Lista de indicadores
for col in colunas_ind:
    df_limpo[col] = df_limpo[col].clip(upper=1.0)                       # Limita valores máximos a 1.0

# Mensagem de sucesso
print("Indicadores proporcionais gerados com sucesso.")

Calculando as dimensões de Saneamento, Educação, Raça e Habitação...
Indicadores proporcionais gerados com sucesso.


In [ ]:
# Aplica normalização invertida para renda e calcula índice de vulnerabilidade de renda
print("Aplicando a Normalização Min-Max Invertida para a Renda...")

# Isola as rendas maiores que zero para encontrar o mínimo real da região
renda_filtrada = df_limpo[df_limpo['V06004'] > 0]['V06004']              # Filtra rendas > 0
renda_maxima = renda_filtrada.max()                                     # Calcula renda máxima
renda_minima = renda_filtrada.min()                                     # Calcula renda mínima

# Aplica a fórmula: se não tem renda (0), ganha risco máximo (1). Se tem, normaliza.
df_limpo['ind_vulnerabilidade_renda'] = np.where(
    df_limpo['V06004'] > 0,                                             # Se renda > 0
    (renda_maxima - df_limpo['V06004']) / (renda_maxima - renda_minima),# Normaliza invertido
    1.0                                                                # Senão, risco máximo
)

# Mensagem de parâmetros
print("Parâmetros encontrados:")
print("-> Renda Máxima na base: R$", renda_maxima)                      # Exibe renda máxima
print("-> Renda Mínima na base: R$", renda_minima)                      # Exibe renda mínima
print("Índice de renda calculado de 0 (melhor) a 1 (pior).")            # Mensagem final

Aplicando a Normalização Min-Max Invertida para a Renda...
Parâmetros encontrados:
-> Renda Máxima na base: R$ 115680.0
-> Renda Mínima na base: R$ 80.0
Índice de renda calculado de 0 (melhor) a 1 (pior).


In [ ]:
# Organiza a estrutura final, filtra colunas relevantes e exporta para CSV e Excel
print("Organizando a estrutura final e gerando os arquivos de saída...") # Mensagem inicial

# Filtra apenas as colunas que importam para o QGIS/Mapeamento
colunas_finais = [
    'CD_SETOR', 'NM_MUN', 'NM_BAIRRO', 'SITUACAO',
    'ind_agua_inadequada', 'ind_esgoto_inadequado', 'ind_lixo_inadequado',
    'ind_analfabetismo', 'ind_cor_raca', 'ind_densidade_habitacional',
    'ind_vulnerabilidade_renda'
 ]

df_ivs_final = df_limpo[colunas_finais].copy()                          # Cria DataFrame final com colunas filtradas

caminho_saida_csv = caminho_bd + 'Base_Analitica_IVS_Calculado.csv'     # Define caminho do arquivo CSV
caminho_saida_excel = caminho_bd + 'Base_Analitica_IVS_Calculado.xlsx'  # Define caminho do arquivo Excel

# Exporta para CSV
df_ivs_final.to_csv(caminho_saida_csv, index=False, sep=';', encoding='utf-8-sig')  # Salva CSV

# Exporta para Excel
with pd.ExcelWriter(caminho_saida_excel, engine='xlsxwriter') as writer:             # Abre writer Excel
    df_ivs_final.to_excel(writer, sheet_name='IVS_Resultados', index=False)          # Salva Excel

print("\nConcluído com excelência! Base analítica final gerada nas extensões CSV e XLSX.")  # Mensagem final

Organizando a estrutura final e gerando os arquivos de saída...

Concluído com excelência! Base analítica final gerada nas extensões CSV e XLSX.
